In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, expr
from pyspark.sql.types import *
from config import KAFKA_BROKER, TOPICS


spark = SparkSession.builder \
    .appName("FlightDelayStream") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Schema الرحلات
flight_schema = StructType([
    StructField("icao24", StringType()),
    StructField("callsign", StringType()),
    StructField("origin_country", StringType()),
    StructField("longitude", DoubleType()),
    StructField("latitude", DoubleType()),
    StructField("altitude", DoubleType()),
    StructField("velocity", DoubleType()),
    StructField("heading", DoubleType()),
    StructField("on_ground", BooleanType()),
    StructField("timestamp", LongType())
])

# Schema الطقس
weather_schema = StructType([
    StructField("airport", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("temperature", DoubleType()),
    StructField("wind_speed", DoubleType()),
    StructField("precipitation", DoubleType()),
    StructField("weathercode", IntegerType()),
    StructField("timestamp", StringType())
])

# اقرأ flights من Kafka
flights_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", TOPICS["flights"]) \
    .option("startingOffsets", "latest") \
    .load() \
    .select(from_json(col("value").cast("string"), flight_schema).alias("data")) \
    .select("data.*")

# اقرأ weather من Kafka
weather_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", TOPICS["weather"]) \
    .option("startingOffsets", "latest") \
    .load() \
    .select(from_json(col("value").cast("string"), weather_schema).alias("data")) \
    .select("data.*")

# اطبع flights
flights_query = flights_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .queryName("flights_stream") \
    .start()

# اطبع weather
weather_query = weather_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .queryName("weather_stream") \
    .start()

print("Consumer started - waiting for data...")
spark.streams.awaitAnyTermination()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/17 01:04:34 WARN Utils: Your hostname, refat, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/17 01:04:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ahmed-refat/.ivy2.5.2/cache
The jars for the packages stored in: /home/ahmed-refat/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c93dc4c6-b3b5-4d20-b7cc-294e6a4cac76;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.0 in central
	found org.apache.kafka#kafka-clients;3.9.1 i

Consumer started - waiting for data...


-------------------------------------------
Batch: 0
-------------------------------------------
-------------------------------------------
Batch: 0
-------------------------------------------
+-------+--------+---------+-----------+----------+-------------+-----------+---------+
|airport|latitude|longitude|temperature|wind_speed|precipitation|weathercode|timestamp|
+-------+--------+---------+-----------+----------+-------------+-----------+---------+
+-------+--------+---------+-----------+----------+-------------+-----------+---------+

+------+--------+--------------+---------+--------+--------+--------+-------+---------+---------+
|icao24|callsign|origin_country|longitude|latitude|altitude|velocity|heading|on_ground|timestamp|
+------+--------+--------------+---------+--------+--------+--------+-------+---------+---------+
+------+--------+--------------+---------+--------+--------+--------+-------+---------+---------+



-------------------------------------------
Batch: 1
-------------------------------------------
-------------------------------------------
Batch: 1
-------------------------------------------
+-------+--------+---------+-----------+----------+-------------+-----------+----------------+
|airport|latitude|longitude|temperature|wind_speed|precipitation|weathercode|timestamp       |
+-------+--------+---------+-----------+----------+-------------+-----------+----------------+
|JFK    |40.6413 |-73.7781 |18.4       |6.21      |0.0          |3          |2026-04-16T23:00|
|LGA    |40.7769 |-73.874  |24.0       |5.76      |0.0          |2          |2026-04-16T23:00|
|EWR    |40.6895 |-74.1745 |30.6       |4.18      |0.0          |0          |2026-04-16T23:00|
+-------+--------+---------+-----------+----------+-------------+-----------+----------------+

+------+--------+--------------+---------+--------+--------+--------+-------+---------+----------+
|icao24|callsign|origin_country|longitude

-------------------------------------------
Batch: 3
-------------------------------------------


...

-------------------------------------------
Batch: 80
-------------------------------------------
+------+--------+--------------+---------+--------+--------+--------+-------+---------+----------+
|icao24|callsign|origin_country|longitude|latitude|altitude|velocity|heading|on_ground|timestamp |
+------+--------+--------------+---------+--------+--------+--------+-------+---------+----------+
|a091d5|DAL724  |United States |-73.8652 |40.7718 |NULL    |0.06    |326.25 |true     |1776416393|
|a64ab0|VJA504  |United States |-74.0641 |40.8545 |NULL    |0.0     |126.56 |true     |1776416367|
|a4414a|DAL758  |United States |-73.7001 |40.7151 |518.16  |91.58   |210.74 |false    |1776416400|
|a080b8|UPS1114 |United States |-73.7711 |40.6293 |NULL    |6.17    |216.56 |true     |1776416400|
|a06e11|DAL2115 |United States |-73.8591 |40.7685 |NULL    |0.06    |132.19 |true     |1776416394|
|aa6ccb|AAL10   |United States |-74.4419 |40.9457 |6370.32 |226.5   |98.88  |false    |1776416399|
|ace62a|FDX

26/04/17 11:00:15 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:707)
	at org.apache.spark.storage.BlockManagerMasterE

-------------------------------------------
Batch: 73
-------------------------------------------
+-------+--------+---------+-----------+----------+-------------+-----------+----------------+
|airport|latitude|longitude|temperature|wind_speed|precipitation|weathercode|timestamp       |
+-------+--------+---------+-----------+----------+-------------+-----------+----------------+
|JFK    |40.6413 |-73.7781 |15.5       |1.66      |0.0          |3          |2026-04-17T09:00|
|LGA    |40.7769 |-73.874  |19.3       |0.5       |0.0          |3          |2026-04-17T09:00|
|EWR    |40.6895 |-74.1745 |20.4       |1.75      |0.0          |3          |2026-04-17T09:00|
+-------+--------+---------+-----------+----------+-------------+-----------+----------------+



26/04/17 11:00:25 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:707)
	at org.apache.spark.storage.BlockManagerMasterE

ConnectionRefusedError: [Errno 111] Connection refused

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, udf, when, broadcast
from pyspark.sql.types import *
import joblib
import numpy as np
import pandas as pd
import requests

KAFKA_BROKER = "localhost:9092"
# كتابه اسماء ال topics في كافكا 
TOPICS = {
    "flights": "flight_raw",
    "weather": "weather_raw"
}

#------------------------------------------
# تحميل المودل من اللوكل 
MODEL_PATH = "/home/ahmed-refat/Desktop/flights & airports/ML model"
model = joblib.load(f"{MODEL_PATH}/xgboost_delay_model.pkl")
feature_names = joblib.load(f"{MODEL_PATH}/feature_names.pkl")

print(f"Model loaded!")
print(f"Features: {feature_names}")

#-----------------------------------------------------------
# create sparksession 
spark = SparkSession.builder \
    .appName("FlightDelayStream") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark ready!")

#----------------------------------------------------------------
# write schema for flight stream 
flight_schema = StructType([
    StructField("icao24", StringType()),
    StructField("callsign", StringType()),
    StructField("origin_country", StringType()),
    StructField("longitude", DoubleType()),
    StructField("latitude", DoubleType()),
    StructField("altitude", DoubleType()),
    StructField("velocity", DoubleType()),
    StructField("heading", DoubleType()),
    StructField("on_ground", BooleanType()),
    StructField("timestamp", LongType())
])

#----------------------------------------------
# airline mapping من callsign
airline_mapping = {
    "DAL": "DL", "AAL": "AA", "UAL": "UA",
    "SWA": "WN", "JBU": "B6", "NKS": "NK",
    "RPA": "YX", "EDV": "9E", "GJS": "G7",
    "LXJ": "XO", "EJA": "XO", "BAW": "BA",
    "AIC": "AI"
}

airline_codes = list(set(airline_mapping.values()))
airline_to_int = {a: i for i, a in enumerate(airline_codes)}

def predict_delay(callsign, lat, lon,
                  temperature, wind_speed, wind_gust,
                  precipitation, visibility, humidity, cloudcover):
    try:
        # استخرج airline code من callsign
        prefix = callsign[:3] if callsign else "UNK"
        airline = airline_mapping.get(prefix, "UNK")
        airline_int = airline_to_int.get(airline, -1)

        features = pd.DataFrame([[
            airline_int, lat, lon,
            temperature, wind_speed, wind_gust,
            precipitation, visibility, humidity, cloudcover
        ]], columns=feature_names)

        pred = model.predict(features)[0]
        return int(pred)
    except:
        return -1

predict_udf = udf(predict_delay, IntegerType())

#------------------------------------------
# جيب الـ weather مرة واحدة كـ static من API مباشرة
# عشان نتجنب مشكلة stream-stream join
def get_weather_static():
    NY_AIRPORTS = {
        "JFK": {"lat": 40.6413, "lon": -73.7781},
        "LGA": {"lat": 40.7769, "lon": -73.8740},
        "EWR": {"lat": 40.6895, "lon": -74.1745}
    }
    records = []
    for airport, coords in NY_AIRPORTS.items():
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": coords["lat"],
            "longitude": coords["lon"],
            "current": "temperature_2m,wind_speed_10m,wind_gusts_10m,precipitation,visibility,relative_humidity_2m,cloud_cover",
            "wind_speed_unit": "ms"
        }
        res = requests.get(url, params=params).json()
        current = res["current"]
        records.append({
            "airport": airport,
            "weather_lat": coords["lat"],
            "weather_lon": coords["lon"],
            "temperature": float(current["temperature_2m"]),
            "wind_speed": float(current["wind_speed_10m"]),
            "wind_gust": float(current["wind_gusts_10m"]),
            "precipitation": float(current["precipitation"]),
            "visibility": float(current.get("visibility", 10000.0)),
            "humidity": float(current["relative_humidity_2m"]),
            "cloudcover": float(current["cloud_cover"])
        })
    return records

# تحويل الـ weather لـ Spark static DataFrame
weather_records = get_weather_static()
weather_static = spark.createDataFrame(pd.DataFrame(weather_records))
print("Weather static loaded!")
weather_static.show()

#-------------------------------------------------------------
# القراءه من كافكا - flights فقط
# الـ weather بقا static مش stream
flights_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", TOPICS["flights"]) \
    .option("startingOffsets", "latest") \
    .load() \
    .select(from_json(col("value").cast("string"), flight_schema).alias("data")) \
    .select("data.*")

print("Stream ready!")

#------------------------------
# rename عشان منتلخبطش بعد الجوين
flights_renamed = flights_df.select(
    col("callsign"),
    col("origin_country"),
    col("latitude").alias("flight_lat"),
    col("longitude").alias("flight_lon"),
    col("altitude"),
    col("velocity"),
    col("on_ground"),
    col("timestamp").alias("flight_timestamp")
)

#-------------------------------------------------------------
# join بين flights stream و weather static
# broadcast عشان الـ weather صغير (3 rows بس)
joined_df = flights_renamed.join(
    broadcast(weather_static),
    (flights_renamed.flight_lon.between(
        weather_static.weather_lon - 0.5,
        weather_static.weather_lon + 0.5
    )) &
    (flights_renamed.flight_lat.between(
        weather_static.weather_lat - 0.5,
        weather_static.weather_lat + 0.5
    )),
    "left"
)

#-------------------------------------------------------------
# تطبيق المودل وإضافة عمود is_delayed
result_df = joined_df.withColumn(
    "is_delayed",
    predict_udf(
        col("callsign"),
        col("flight_lat"),
        col("flight_lon"),
        col("temperature"),
        col("wind_speed"),
        col("wind_gust"),
        col("precipitation"),
        col("visibility"),
        col("humidity"),
        col("cloudcover")
    )
).select(
    col("callsign"),
    col("origin_country"),
    col("flight_lat"),
    col("flight_lon"),
    col("on_ground"),
    col("temperature"),
    col("wind_speed"),
    col("is_delayed"),
    when(col("is_delayed") == 1, "DELAYED")
    .when(col("is_delayed") == 0, "ON TIME")
    .otherwise("UNKNOWN").alias("status")
)

#------------------------------------------------------------
# طباعة النتيجة النهائية في الكونسول
query = result_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .option("numRows", 20) \
    .option("checkpointLocation", "/tmp/checkpoint/prediction") \
    .queryName("prediction_stream") \
    .start()

print("Prediction stream started...")
spark.streams.awaitAnyTermination()

Model loaded!
Features: ['airline', 'airport_lat', 'airport_lon', 'temperature', 'wind_speed', 'wind_gust', 'precipitation', 'visibility', 'humidity', 'cloudcover']


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/24 04:33:34 WARN Utils: Your hostname, refat, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/24 04:33:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ahmed-refat/.ivy2.5.2/cache
The jars for the packages stored in: /home/ahmed-refat/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e685e104-3ac6-48ab-856f-c77c5d1fc063;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.0 in central
	found org.apache.kafka#kafka-clients;3.9.1 i

Spark ready!
Weather static loaded!


+-------+-----------+-----------+-----------+----------+---------+-------------+----------+--------+----------+
|airport|weather_lat|weather_lon|temperature|wind_speed|wind_gust|precipitation|visibility|humidity|cloudcover|
+-------+-----------+-----------+-----------+----------+---------+-------------+----------+--------+----------+
|    JFK|    40.6413|   -73.7781|       13.4|      1.44|      2.2|          0.0|   19500.0|    80.0|     100.0|
|    LGA|    40.7769|    -73.874|       19.3|      3.76|      6.7|          0.0|   72400.0|    27.0|     100.0|
|    EWR|    40.6895|   -74.1745|       19.8|      2.01|      6.6|          0.0|   73800.0|    24.0|     100.0|
+-------+-----------+-----------+-----------+----------+---------+-------------+----------+--------+----------+

Stream ready!
Prediction stream started...


-------------------------------------------
Batch: 0
-------------------------------------------
+--------+--------------+----------+----------+---------+-----------+----------+----------+------+
|callsign|origin_country|flight_lat|flight_lon|on_ground|temperature|wind_speed|is_delayed|status|
+--------+--------------+----------+----------+---------+-----------+----------+----------+------+
+--------+--------------+----------+----------+---------+-----------+----------+----------+------+



...

-------------------------------------------
Batch: 2
-------------------------------------------


+--------+--------------+----------+----------+---------+-----------+----------+----------+-------+
|callsign|origin_country|flight_lat|flight_lon|on_ground|temperature|wind_speed|is_delayed|status |
+--------+--------------+----------+----------+---------+-----------+----------+----------+-------+
|DAL2513 |United States |40.77     |-73.8593  |true     |13.4       |1.44      |0         |ON TIME|
|DAL2513 |United States |40.77     |-73.8593  |true     |19.3       |3.76      |0         |ON TIME|
|DAL2513 |United States |40.77     |-73.8593  |true     |19.8       |2.01      |1         |DELAYED|
|EDV5111 |United States |40.63     |-73.7761  |true     |13.4       |1.44      |0         |ON TIME|
|EDV5111 |United States |40.63     |-73.7761  |true     |19.3       |3.76      |0         |ON TIME|
|EDV5111 |United States |40.63     |-73.7761  |true     |19.8       |2.01      |0         |ON TIME|
|RPA3658 |United States |40.697    |-74.1706  |true     |13.4       |1.44      |0         |ON TIME|


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
RuntimeError: reentrant call inside <_io.BufferedReader name=79>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/site-packages/py4j/clientserver.py", line 566, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/site-packages/py4j/clientserver.py

Py4JError: An error occurred while calling o214.awaitAnyTermination

26/04/24 04:36:45 ERROR Executor: Exception in task 0.0 in stage 8.0 (TID 11)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3301, in main
    split_index = read_int(infile)
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 595, in read_int
    length = stream.read(4)
KeyboardInterrupt

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:645)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1029)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1014)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:596)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	at